# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 8.9 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict
from onnx import shape_inference

In [5]:
TASK_ID='task113'
CH,H,W=10,30,30
TASK_H=10
ROOT=Path.cwd()
TASK_PATH=Path(COMPETITION)/'task113.json'
if not TASK_PATH.exists():
    TASK_PATH=Path('/mnt/data/task113.json')
ONNX_PATH=ROOT / f'{TASK_ID}_zero_pad_static_graph.onnx'
ZIP_PATH=ROOT / 'submission.zip'
TASK_ZIP_PATH=ROOT / f'{TASK_ID}_zero_pad_submission.zip'
STATIC_ZIP_PATH=ROOT / f'{TASK_ID}_zero_pad_static_graph_submission.zip'
AUDIT_JSON=ROOT / f'{TASK_ID}_zero_pad_audit.json'
AUDIT_CSV=ROOT / f'{TASK_ID}_zero_pad_audit.csv'

In [6]:
class Task113ZeroPad(nn.Module):
    def forward(self, x):
        # x is one-hot inside the real grid and all-zero outside it.
        active = torch.clamp(x.sum(dim=1, keepdim=True), 0.0, 1.0)
        core = x[:,1:,:,:]
        top = core[:,:,:TASK_H,:]
        mirror = torch.flip(top, dims=[2])
        fg_top = torch.clamp(top + mirror, 0.0, 1.0)
        fg_rest = core[:,:,TASK_H:,:] * 0.0
        fg = torch.cat([fg_top, fg_rest], dim=2) * active
        bg = active * (1.0 - torch.clamp(fg.sum(dim=1, keepdim=True), 0.0, 1.0))
        return torch.cat([bg, fg], dim=1)

def grid_to_tensor(grid):
    # Correct Kaggle/reference encoding: no background channel in padded area.
    x=np.zeros((1,CH,H,W), dtype=np.float32)
    a=np.array(grid, dtype=np.int64)
    h,w=a.shape
    for c in range(CH):
        x[0,c,:h,:w]=(a==c)
    return x

def output_to_tensor(grid):
    return grid_to_tensor(grid)

def pred_grid(y, h, w):
    return y[0,:,:h,:w].argmax(axis=0).astype(np.int64)

def exact_raw_eval(sess, examples):
    inp=sess.get_inputs()[0].name
    exact=0
    raw=0
    for ex in examples:
        x=grid_to_tensor(ex['input'])
        y=sess.run(None,{inp:x})[0]
        tgt=output_to_tensor(ex['output'])
        if np.array_equal(y.round().astype(np.float32), tgt):
            raw += 1
        h,w=np.array(ex['output']).shape
        if np.array_equal(pred_grid(y,h,w), np.array(ex['output'], dtype=np.int64)):
            exact += 1
    return {'argmax_exact': exact, 'raw_exact': raw, 'total': len(examples)}

def onnx_shape(vi):
    return [int(d.dim_value) for d in vi.type.tensor_type.shape.dim]

def check_raw_contract(sess, examples, n=20):
    inp=sess.get_inputs()[0].name
    for ex in examples[:n]:
        x=grid_to_tensor(ex['input'])
        y=sess.run(None,{inp:x})[0]
        active=x.sum(axis=1, keepdims=True)
        sums=y.sum(axis=1, keepdims=True)
        vals=np.unique(np.round(y,5))
        if not set(vals.tolist()).issubset({0.0,1.0}):
            return False, f'bad values {vals}'
        if not np.array_equal(np.round(sums,5), active):
            return False, 'channel sum not active-mask'
    return True, 'ok'

# OOD width/color checks for the row-mirroring rule.
def make_synth(width, colors):
    g=np.zeros((10,width), dtype=np.int64)
    for r,c in colors:
        g[r,:]=c
    out=g.copy()
    for r in range(10):
        rr=9-r
        mask=(g[rr]!=0)
        out[r,mask]=g[rr,mask]
    return {'input':g.tolist(), 'output':out.tolist()}

synth=[]
for w in range(1,31):
    synth.append(make_synth(w, [(0,2),(1,2),(2,3)]))
    synth.append(make_synth(w, [(0,8),(1,5)]))
    synth.append(make_synth(w, [(0,7),(1,7),(2,6),(3,6)]))

In [7]:
model=Task113ZeroPad().eval()
dummy=torch.zeros(1,CH,H,W,dtype=torch.float32)
dummy[:,0,:TASK_H,:6]=1.0
torch.onnx.export(model,dummy,str(ONNX_PATH),input_names=['input'],output_names=['output'],opset_version=13,dynamic_axes=None,do_constant_folding=True,dynamo=False)
m=onnx.load(str(ONNX_PATH))
for vi in [m.graph.input[0], m.graph.output[0]]:
    for d,v in zip(vi.type.tensor_type.shape.dim,[1,CH,H,W]):
        d.dim_param=''; d.dim_value=int(v)
onnx.save(m,str(ONNX_PATH))
onnx.checker.check_model(m)
ops=dict(collections.Counter(n.op_type for n in m.graph.node))
print('shape', onnx_shape(m.graph.input[0]), onnx_shape(m.graph.output[0]))
print('size', ONNX_PATH.stat().st_size)
print('ops', ops)

shape [1, 10, 30, 30] [1, 10, 30, 30]
size 3503
ops {'Constant': 25, 'ReduceSum': 2, 'Clip': 3, 'Slice': 4, 'Add': 1, 'Mul': 3, 'Concat': 2, 'Sub': 1}


/tmp/ipykernel_16/200299285.py:4: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model,dummy,str(ONNX_PATH),input_names=['input'],output_names=['output'],opset_version=13,dynamic_axes=None,do_constant_folding=True,dynamo=False)


In [8]:
task=json.load(open(TASK_PATH))
sess=ort.InferenceSession(str(ONNX_PATH), providers=['CPUExecutionProvider'])
arc=task.get('arc-gen', [])
holdout_start=int(len(arc)*0.4)
contract_ok, contract_msg = check_raw_contract(sess, task['train']+task['test']+arc)
health={
    'task': TASK_ID,
    'input_shape': onnx_shape(onnx.load(str(ONNX_PATH)).graph.input[0]),
    'output_shape': onnx_shape(onnx.load(str(ONNX_PATH)).graph.output[0]),
    'size_bytes': ONNX_PATH.stat().st_size,
    'op_counts': ops,
    'forbidden_ops_present': sorted(set(['Loop','Scan','NonZero','Unique','Script','Function']).intersection(ops)),
    'risky_ops_present': sorted(set(['Shape','Gather','ConstantOfShape','Expand','Range','ScatterND']).intersection(ops)),
    'raw_zero_padding_contract': {'ok': contract_ok, 'message': contract_msg},
    'train': exact_raw_eval(sess,task['train']),
    'test': exact_raw_eval(sess,task['test']),
    'arc_all': exact_raw_eval(sess,arc),
    'arc_holdout_60pct': exact_raw_eval(sess,arc[holdout_start:]),
    'synthetic_ood': exact_raw_eval(sess,synth),
}
print(json.dumps(health, indent=2))
assert health['input_shape']==[1,10,30,30] and health['output_shape']==[1,10,30,30]
assert health['size_bytes'] < 1_400_000
assert not health['forbidden_ops_present']
assert not health['risky_ops_present']
assert contract_ok
for key in ['train','test','arc_all','arc_holdout_60pct','synthetic_ood']:
    assert health[key]['raw_exact']==health[key]['total'], key
with open(AUDIT_JSON,'w') as f: json.dump(health,f,indent=2)
with open(AUDIT_CSV,'w') as f:
    f.write('task,check,raw_exact,total\n')
    for k in ['train','test','arc_all','arc_holdout_60pct','synthetic_ood']:
        f.write(f"{TASK_ID},{k},{health[k]['raw_exact']},{health[k]['total']}\n")

{
  "task": "task113",
  "input_shape": [
    1,
    10,
    30,
    30
  ],
  "output_shape": [
    1,
    10,
    30,
    30
  ],
  "size_bytes": 3503,
  "op_counts": {
    "Constant": 25,
    "ReduceSum": 2,
    "Clip": 3,
    "Slice": 4,
    "Add": 1,
    "Mul": 3,
    "Concat": 2,
    "Sub": 1
  },
  "forbidden_ops_present": [],
  "risky_ops_present": [],
  "raw_zero_padding_contract": {
    "ok": true,
    "message": "ok"
  },
  "train": {
    "argmax_exact": 2,
    "raw_exact": 2,
    "total": 2
  },
  "test": {
    "argmax_exact": 1,
    "raw_exact": 1,
    "total": 1
  },
  "arc_all": {
    "argmax_exact": 262,
    "raw_exact": 262,
    "total": 262
  },
  "arc_holdout_60pct": {
    "argmax_exact": 158,
    "raw_exact": 158,
    "total": 158
  },
  "synthetic_ood": {
    "argmax_exact": 90,
    "raw_exact": 90,
    "total": 90
  }
}


In [9]:
for zpath in [ZIP_PATH, TASK_ZIP_PATH, STATIC_ZIP_PATH]:
    with zipfile.ZipFile(zpath,'w',zipfile.ZIP_DEFLATED) as z:
        z.write(ONNX_PATH, f'{TASK_ID}.onnx')
print('wrote', ZIP_PATH, TASK_ZIP_PATH, STATIC_ZIP_PATH)

wrote /kaggle/working/submission.zip /kaggle/working/task113_zero_pad_submission.zip /kaggle/working/task113_zero_pad_static_graph_submission.zip
